In [26]:
import numpy as np
import rasterio
from rasterio.warp import reproject, transform_bounds
from rasterio.enums import Resampling
from rasterio.windows import Window
from rasterio.windows import transform as window_transform
import xarray as xr
import rioxarray 
from tqdm import tqdm
from itertools import product


from rasterio.transform import array_bounds
import rasterio.windows

In [2]:
from pathlib import Path

root = Path.cwd()

In [27]:
def template_from_s2(filepath, crs=None, x_dim="x", y_dim="y", time_dim="time", band_dim="band"):
    """
    Returns (dst_crs, dst_transform, dst_height, dst_width)

    Assumes the NetCDF has x/y coordinate variables (projected) OR you provide crs.
    """
    if str(filepath).endswith('nc'):
        da = xr.open_dataarray(filepath)
    
    elif str(filepath).endswith('zarr'):
        da = xr.open_zarr(filepath)['variables']
  
    # Make sure rioxarray knows which dims are spatial
    da = da.rio.set_spatial_dims(x_dim=x_dim, y_dim=y_dim)

    # If CRS is missing in the NetCDF, supply it (e.g., "EPSG:26918" or "EPSG:32618")
    if da.rio.crs is None:
        da = da.rio.write_crs(crs)

    print(f'epsg: {da.rio.crs}')
    # Use a single slice just to ensure coords are loaded consistently
    da0 = da.isel({time_dim: 0, band_dim: 0}) if (time_dim in da.dims and band_dim in da.dims) else da

    height = da0.sizes[y_dim]
    width  = da0.sizes[x_dim]

    return da.rio.crs, da.rio.transform(), height, width

def canopy_percent_from_lidar_landcover(
    landcover_path,
    out_tif_path,
    ref_data_path,
    canopy_class_values=(1,),
    block=512,
    out_dtype="float32",
):

    t = xr.open_zarr(ref_data_path)
    ref_raster = t.rio.write_crs(26918).rio.set_spatial_dims(x_dim='x',y_dim='y').rio.write_coordinate_system()
    dst_crs = ref_raster.rio.crs
    dst_transform = ref_raster.rio.transform()
    dst_height, dst_width = ref_raster.sizes['y'], ref_raster.sizes['x']
    
    with rasterio.open(landcover_path) as src:
        profile = {
            "driver": "GTiff",
            "height": dst_height,
            "width": dst_width,
            "count": 1,
            "dtype": out_dtype,
            "crs": dst_crs,
            "transform": dst_transform,
            "nodata": np.nan,
            "compress": "deflate",
            "predictor": 2,
            "tiled": True,
            "blockxsize": min(block, dst_width),
            "blockysize": min(block, dst_height),
        }

        src_nodata = src.nodata

        with rasterio.open(out_tif_path, "w", **profile) as dst:
            # for row0 in range(0, dst_height, block):
            #     h = min(block, dst_height - row0)
            #     for col0 in tqdm(range(0, dst_width, block)):
            #         w = min(block, dst_width - col0)
            blocks = list(product(range(0, dst_height, block),range(0, dst_width, block )))
            for row0, col0 in tqdm(blocks, desc='windows'):
                    h = min(block, dst_height - row0)
                    w = min(block, dst_width - col0)
                    win = Window(col0, row0, w, h)
                    win_tr = window_transform(win, dst_transform)

                    # Bounds of this destination window (in dst_crs)
                    dst_bounds = rasterio.windows.bounds(win, dst_transform)

                    # Transform those bounds into source CRS, read only what we need
                    src_bounds = transform_bounds(dst_crs, src.crs, *dst_bounds, densify_pts=21)
                    src_win = rasterio.windows.from_bounds(*src_bounds, transform=src.transform)
                    src_win = src_win.round_offsets().round_lengths()

                    src_data = src.read(1, window=src_win, boundless=True)

                    # Binary canopy mask (0/1) with nodata -> nan
                    canopy = np.isin(src_data,canopy_class_values).astype(np.float32)
                    if src_nodata is not None:
                        canopy = np.where(src_data == src_nodata, np.nan, canopy)

                    frac = np.full((h, w), np.nan, dtype=np.float32)

                    reproject(
                        source=canopy,
                        destination=frac,
                        src_transform=rasterio.windows.transform(src_win, src.transform),
                        src_crs=src.crs,
                        src_nodata=np.nan,
                        dst_transform=win_tr,
                        dst_crs=dst_crs,
                        dst_nodata=np.nan,
                        resampling=Resampling.average,  
                    )

                    dst.write((frac * 100.0).astype(out_dtype), 1, window=win)

    print(f'Tree canopy lidar raster written to {out_tif_path}')

In [24]:
import os
import tempfile
import numpy as np
import rasterio
import xarray as xr
from osgeo import gdal
from tqdm import tqdm

gdal.UseExceptions()


def _binarize_to_percent(src_path, dst_path, class_values,
                         canopy_val=100, nodata_out=255):

    with rasterio.open(src_path) as src:
        prof = src.profile.copy()
        prof.update(dtype="uint8", count=1, nodata=nodata_out,
                    compress="deflate", tiled=True,
                    blockxsize=512, blockysize=512)
        src_nodata = src.nodata
        with rasterio.open(dst_path, "w", **prof) as dst:
            windows = [w for _, w in src.block_windows(1)]
            for win in tqdm(windows, desc="binarize"):
                a = src.read(1, window=win)
                out = np.where(np.isin(a, class_values), canopy_val, 0).astype("uint8")
                if src_nodata is not None:
                    out = np.where(a == src_nodata, nodata_out, out)
                dst.write(out, 1, window=win)


def canopy_percent_from_lidar_landcover_gdal(
    landcover_path,
    out_tif_path,
    ref_data_path,
    canopy_class_values=(1,)
):
    # --- target grid from the reference zarr ---
    t = xr.open_zarr(ref_data_path)
    ref = (t.rio.write_crs(26918)
             .rio.set_spatial_dims(x_dim="x", y_dim="y")
             .rio.write_coordinate_system())
    dst_crs_wkt = ref.rio.crs.to_wkt()
    left, bottom, right, top = ref.rio.bounds()
    dst_width = ref.sizes["x"]
    dst_height = ref.sizes["y"]

    tmpdir = tempfile.mkdtemp(prefix="canopy_")
    binary_tmp = os.path.join(tmpdir, "canopy_binary.tif")
    try:
        # --- step 1: binarize (canopy -> 100, else 0, nodata -> 255) ---
        _binarize_to_percent(landcover_path, binary_tmp, canopy_class_values)

        # --- step 2: reproject + area-average onto the ref grid ---
        # average of {0,100} over valid pixels == percent canopy, straight out.
        gdal.Warp(
            out_tif_path,
            binary_tmp,
            dstSRS=dst_crs_wkt,
            outputBounds=(left, bottom, right, top),   # pin to ref extent
            width=dst_width, height=dst_height,        # ...and ref shape
            resampleAlg="average",
            outputType=gdal.GDT_Float32,
            srcNodata=255,                             # nodata excluded from the average
            dstNodata=float("nan"),
            multithread=True,
            warpOptions=["NUM_THREADS=ALL_CPUS"],
            creationOptions=[
                "TILED=YES", "COMPRESS=DEFLATE", "PREDICTOR=2",
                "BLOCKXSIZE=512", "BLOCKYSIZE=512",
            ],
            callback=gdal.TermProgress_nocb,           # native 0–100% progress bar
        )
    finally:
        try:
            os.remove(binary_tmp)
            os.rmdir(tmpdir)
        except OSError:
            pass

    print(f"Tree canopy lidar raster written to {out_tif_path}")

In [29]:
output_label = 'nyc18'

landcover_path = root / 'data' / 'lidar_landcover' / 'Land_Cover' / 'NYC_2017_LiDAR_LandCover.img'
out_tif_path = root / 'data' / output_label / f'{output_label}_tree_canopy_lidar_test.tif'
ref_data_path = root / 'data' / output_label / f'{output_label}_seasonalstats.zarr'

canopy_percent_from_lidar_landcover(landcover_path=landcover_path,out_tif_path=out_tif_path,ref_data_path=ref_data_path)

windows:   2%|▏         | 2/100 [00:38<31:17, 19.16s/it]


KeyboardInterrupt: 

In [20]:
output_label = 'nyc18'

landcover_path = root / 'data' / 'lidar_landcover' / 'Land_Cover' / 'NYC_2017_LiDAR_LandCover.img'
out_tif_path = root / 'data' / output_label / f'{output_label}_tree_canopy_lidar.tif'
ref_data_path = root / 'data' / output_label / f'{output_label}_seasonalstats.zarr'

canopy_percent_from_lidar_landcover(landcover_path=landcover_path,out_tif_path=out_tif_path,ref_data_path=ref_data_path)

100%|██████████| 10/10 [00:29<00:00,  2.98s/it]

Tree canopy lidar raster written to c:\Users\roseh\Documents\utc_paper\data\nyc18\nyc18_tree_canopy_lidar.tif


In [ ]:
## check alignment of outputs
features = xr.open_zarr(ref_data_path)
lidar_raster = rioxarray.open_rasterio(out_tif_path)

print(features.sizes, lidar_raster.shape)          # y,x must match exactly
print(features.dims)                               # must be ('band','y','x'), not transposed
print(features.y.values[:3], lidar_raster.y.values[:3])  # same grid, not just same count
print(features.x.values[:3], lidar_raster.x.values[:3])

Frozen({'band': 180, 'y': 4686, 'x': 4670}) (1, 4686, 4670)
FrozenMappingWarningOnValuesAccess({'band': 180, 'y': 4686, 'x': 4670})
[4529950. 4529940. 4529930.] [4529950. 4529940. 4529930.]
[563070. 563080. 563090.] [563070. 563080. 563090.]


In [ ]:
siteyear = 'nyc21'
year = 2021

lc_path = r'C:\Users\roseh\Downloads\Land_Cover\Land_Cover\NYC_2017_LiDAR_LandCover.img'
#lc_path = root / 'data' / 'nyc17' / f'landcover_nyc_2017_6in_orig.tif'
out_tif_path = root / 'data' / siteyear / f'{siteyear}_tree_canopy_lidar.tif'
filepath = root / 'data' / siteyear/ f'{siteyear}_allbandsandindices.zarr'
dst_crs, dst_transform, dst_h, dst_w = template_from_s2(filepath=filepath)

epsg: EPSG:26918


In [ ]:
da = xr.open_zarr(filepath)['variables']

da.chunks

In [ ]:
out = canopy_percent_to_template(landcover_path=lc_path,out_tif_path=out_tif_path,canopy_class_value=1,dst_crs=dst_crs,dst_transform=dst_transform,dst_height=dst_h,dst_width=dst_w)

In [4]:
lc_path = r'C:\Users\roseh\Downloads\Land_Cover\Land_Cover\NYC_2017_LiDAR_LandCover.img'

siteyear = 'nyc18'
lc_class = 'all_impervious'
lc_class_values = [5,6,7]

out_tif_path = root / 'data' / siteyear / f'{siteyear}_{lc_class}_lidar.tif'
filepath = root / 'model_outputs' / f'{siteyear}_from_nyc21_medians_only_model_canopy_percent.tif'

ref_raster = rioxarray.open_rasterio(filepath)
dst_crs = ref_raster.rio.crs
dst_transform = ref_raster.rio.transform()
dst_h, dst_w = ref_raster.sizes['y'], ref_raster.sizes['x']

out = canopy_percent_to_template(landcover_path=lc_path,out_tif_path=out_tif_path,canopy_class_values=lc_class_values,dst_crs=dst_crs,dst_transform=dst_transform,dst_height=dst_h,dst_width=dst_w)